Image Cropper
============
Short Description:
This script extracts 128x128 image patches centered on each labeled region (bubble) from input images using the corresponding mask files, and saves them as individual cropped images.

To Use with a Different Dataset or Output Location, Change These:

1. Input image folder:
   IMG_DIR = r"path/to/input/images"
   → Folder with the full-size input images.

2. Mask image folder:
   MASK_DIR = r"path/to/pixel/masks"
   → Folder with mask images (.png) indicating where the bubbles are.

3. Output folder for cropped patches:
   OUT_DIR = r"path/to/save/location"
   → This is where the 128x128 cropped patches will be saved.

4. Patch size (if needed):
   PATCH = 128
   → Change this if you want a different output crop size.

What it does:
- Loads each image and its corresponding binary mask.
- Finds all connected regions (bubbles) in the mask.
- Calculates the centroid of each region.
- Crops a 128x128 patch centered on each centroid (with reflect-padding if needed).
- Saves each cropped patch as a separate PNG in the output folder.


In [ ]:
import os
import re
import cv2
import numpy as np

# ── 1) CONFIGURE YOUR PATHS ─────────────────────────────────────────────
IMG_DIR  = r"C:/BLENDER/BubbleID/Code/Bubblina/Validation Frames"        
MASK_DIR = r"C:/BLENDER/BubbleID/Code/Bubblina/PixelLabelData"  
OUT_DIR  = r"C:/BLENDER/BubbleID/Code/CNN/Cropped128_Bubblina_val_Bubble"  
os.makedirs(OUT_DIR, exist_ok=True)

# ── 2) PARAMETERS ────────────────────────────────────────────────────────
PATCH = 128                # output patch size
HALF  = PATCH // 2         # half‐width, for centering

# ── 3) HELPERS ───────────────────────────────────────────────────────────
def crop_with_reflect(frame, cx, cy, size=PATCH):
    """
    Extract a size×size crop centered on (cx,cy).
    If the window runs off the image borders, reflect‐pad as needed.
    """
    h, w = frame.shape[:2]
    x0 = int(cx - size//2)
    y0 = int(cy - size//2)
    x1 = x0 + size
    y1 = y0 + size

    # compute needed padding
    pad_left   = max(0, -x0)
    pad_top    = max(0, -y0)
    pad_right  = max(0, x1 - w)
    pad_bottom = max(0, y1 - h)

    # reflect‐pad if any side goes out of bounds
    if pad_left or pad_top or pad_right or pad_bottom:
        frame = cv2.copyMakeBorder(
            frame,
            pad_top, pad_bottom,
            pad_left, pad_right,
            borderType=cv2.BORDER_REFLECT_101
        )
        x0 += pad_left
        x1 += pad_left
        y0 += pad_top
        y1 += pad_top

    return frame[y0:y1, x0:x1]

# ── 4) MAIN LOOP ────────────────────────────────────────────────────────
# regex to pull "frame_3750" from "Label_1_frame_3750.png"
pattern = re.compile(r"^Label_\d+_(frame_\d+)\.png$", re.IGNORECASE)


for mask_fn in sorted(os.listdir(MASK_DIR)):
    # only process PNG mask files
    if not mask_fn.lower().endswith('.png'):
        continue

    m = pattern.match(mask_fn)
    if not m:
        print(f"skipping unexpected filename: {mask_fn}")
        continue

    stem = m.group(1)  # e.g. "FancyAnt_80_frame_0002"
    img_path  = os.path.join(IMG_DIR,  stem + ".jpg")
    mask_path = os.path.join(MASK_DIR, mask_fn)

    # check existence
    if not os.path.isfile(img_path):
        print(f"missing frame for {stem}, skipping")
        continue

    # load frame & mask
    img  = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        print(f"could not load frame or mask for {stem}")
        continue

    # binarize mask (bubble pixels > 0)
    bin_mask = (mask > 0).astype(np.uint8)

    # find each connected bubble contour
    contours, _ = cv2.findContours(
        bin_mask, 
        cv2.RETR_EXTERNAL, 
        cv2.CHAIN_APPROX_SIMPLE
    )

    # crop one patch per bubble
    for idx, cnt in enumerate(contours, start=1):
        # compute centroid via image moments
        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        # reflect‐pad & crop
        patch = crop_with_reflect(img, cx, cy, PATCH)

        # save as PNG: <stem>_1.png, <stem>_2.png, …
        out_name = f"{stem}_{idx}.png"
        cv2.imwrite(os.path.join(OUT_DIR, out_name), patch)

print("Done cropping positives →", OUT_DIR)